# Creación y ajuste del dataset del TFM

## Merge de los datasets

In [1]:
import pandas as pd
import os

data_path = "data"

for file in os.listdir(data_path):
    
    file_path = os.path.join(data_path, file)
    
    try:
        if file.endswith(".csv"):
            df = pd.read_csv(file_path, nrows=5)
            
        elif file.endswith(".xpt"):
            df = pd.read_sas(file_path)
            
        else:
            continue
        
        print("\n==============================")
        print(f"Dataset: {file}")
        print("Número de columnas:", len(df.columns))
        print("Columnas:")
        print(list(df.columns))
        
    except Exception as e:
        print(f"Error leyendo {file}: {e}")

C:\Users\Usuario\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (



Dataset: BAX_L.xpt
Número de columnas: 45
Columnas:
['SEQN', 'BAXMSTAT', 'BAXRXNC', 'BAXRXND', 'BAX5STAT', 'BAQ110', 'BAQ121', 'BAQ125', 'BAQ132', 'BAQ140', 'BAQ150', 'BAQ160', 'BAQ170', 'BAQ201', 'BAQ173', 'BAXPF11', 'BAXTC11', 'BAARFC11', 'BAXPF12', 'BAXTC12', 'BAARFC12', 'BAXPF21', 'BAXTC21', 'BAARFC21', 'BAXPF22', 'BAXTC22', 'BAARFC22', 'BAXPF31', 'BAXTC31', 'BAARFC31', 'BAXPF32', 'BAXTC32', 'BAARFC32', 'BAXPF41', 'BAXTC41', 'BAARFC41', 'BAXPF42', 'BAXTC42', 'BAARFC42', 'BAXPF51', 'BAXTC51', 'BAARFC51', 'BAXPF52', 'BAXTC52', 'BAARFC52']

Dataset: BMX_L.xpt
Número de columnas: 22
Columnas:
['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXHIP', 'BMIHIP']

Dataset: BPXO_L.xpt
Número de columnas: 12
Columnas:
['SEQN', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1', 'BPX

Vamos a tener que cambiar el nombre de las columnas para saber con qué estamos realmente trabajando, dado que los nombres actuales no son nada intuitivos y pueden llevar a confusiones importantes en apartados futuros.

## Merge datasets

In [3]:
# -----------------------------
# 1. Cargar todos los .xpt
# -----------------------------
datasets = {}

for file in os.listdir(data_path):
    if file.lower().endswith(".xpt"):
        path = os.path.join(data_path, file)
        df = pd.read_sas(path)
        
        # normalizar nombres de columnas a str
        df.columns = df.columns.astype(str)
        
        # guardar usando el nombre del archivo sin extensión
        name = os.path.splitext(file)[0]
        datasets[name] = df

print("Datasets cargados:")
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Datasets cargados:
BAX_L: (4771, 45)
BMX_L: (8860, 22)
BPXO_L: (7801, 12)
DEMO_L: (11933, 27)
DPQ_L: (6337, 11)
DSQTOT_L: (8860, 40)
LUX_L: (7199, 13)
PAQ_L: (8153, 8)
RHQ_L: (3917, 13)
RXQ_RX_L: (11933, 3)
SLQ_L: (8501, 7)


In [4]:
# -----------------------------
# 2. Comprobar unicidad de SEQN
# -----------------------------
print("\n--- Comprobación de SEQN único por dataset ---")
for name, df in datasets.items():
    if "SEQN" not in df.columns:
        print(f"{name}: NO tiene SEQN")
        continue
    
    n_rows = len(df)
    n_unique = df["SEQN"].nunique()
    duplicated = n_rows - n_unique
    
    if duplicated == 0:
        print(f"{name}: OK, 1 fila por persona")
    else:
        print(f"{name}: OJO, {duplicated} filas extra por SEQN (dataset longitudinal / múltiples registros)")


--- Comprobación de SEQN único por dataset ---
BAX_L: OK, 1 fila por persona
BMX_L: OK, 1 fila por persona
BPXO_L: OK, 1 fila por persona
DEMO_L: OK, 1 fila por persona
DPQ_L: OK, 1 fila por persona
DSQTOT_L: OK, 1 fila por persona
LUX_L: OK, 1 fila por persona
PAQ_L: OK, 1 fila por persona
RHQ_L: OK, 1 fila por persona
RXQ_RX_L: OK, 1 fila por persona
SLQ_L: OK, 1 fila por persona


In [5]:
# -----------------------------
# 3. Elegir DEMO como base
# -----------------------------
base_name = "DEMO_L"
merged = datasets[base_name].copy()

print(f"\nBase inicial: {base_name} -> {merged.shape}")


Base inicial: DEMO_L -> (11933, 27)


In [6]:
# -----------------------------
# 4. Función para agregar datasets con múltiples filas por SEQN
# -----------------------------
def aggregate_by_seqn(df, dataset_name):
    """
    Convierte un dataset con varias filas por SEQN en uno de 1 fila por persona.
    Estrategia:
    - variables numéricas: count, mean, max
    - variables categóricas/texto: número de valores únicos
    """
    df = df.copy()
    
    # excluir SEQN
    cols = [c for c in df.columns if c != "SEQN"]
    
    numeric_cols = df[cols].select_dtypes(include="number").columns.tolist()
    other_cols = [c for c in cols if c not in numeric_cols]
    
    agg_dict = {}
    
    for col in numeric_cols:
        agg_dict[col] = ["count", "mean", "max"]
    
    for col in other_cols:
        agg_dict[col] = ["nunique"]
    
    out = df.groupby("SEQN").agg(agg_dict)
    
    # aplanar nombres de columnas
    out.columns = [
        f"{dataset_name}__{col}__{stat}"
        for col, stat in out.columns
    ]
    
    out = out.reset_index()
    return out

In [7]:
# -----------------------------
# 5. Merge progresivo
# -----------------------------
for name, df in datasets.items():
    if name == base_name:
        continue
    
    if "SEQN" not in df.columns:
        print(f"Saltando {name}: no tiene SEQN")
        continue

    # comprobar si hay varias filas por persona
    if df["SEQN"].nunique() == len(df):
        # merge directo
        # renombrar columnas (excepto SEQN) para evitar conflictos
        rename_map = {
            col: f"{name}__{col}"
            for col in df.columns
            if col != "SEQN"
        }
        df_renamed = df.rename(columns=rename_map)
        merged = merged.merge(df_renamed, on="SEQN", how="left")
        print(f"Merge directo con {name}: {merged.shape}")
    else:
        # agregar antes de unir
        df_agg = aggregate_by_seqn(df, name)
        merged = merged.merge(df_agg, on="SEQN", how="left")
        print(f"Merge con agregación previa de {name}: {merged.shape}")

Merge directo con BAX_L: (11933, 71)
Merge directo con BMX_L: (11933, 92)
Merge directo con BPXO_L: (11933, 103)
Merge directo con DPQ_L: (11933, 113)
Merge directo con DSQTOT_L: (11933, 152)
Merge directo con LUX_L: (11933, 164)
Merge directo con PAQ_L: (11933, 171)
Merge directo con RHQ_L: (11933, 183)
Merge directo con RXQ_RX_L: (11933, 185)
Merge directo con SLQ_L: (11933, 191)


In [8]:
# -----------------------------
# 6. Revisar resultado final
# -----------------------------
print("\n--- Dataset final ---")
print("Shape:", merged.shape)
print("Número de personas únicas:", merged["SEQN"].nunique())
print("Número de columnas:", len(merged.columns))

# comprobar duplicados de SEQN en el final
dup_final = merged["SEQN"].duplicated().sum()
print("Duplicados de SEQN en dataset final:", dup_final)


--- Dataset final ---
Shape: (11933, 191)
Número de personas únicas: 11933
Número de columnas: 191
Duplicados de SEQN en dataset final: 0


In [ ]:
# -----------------------------
# 7. Guardar dataset final
# -----------------------------
merged.to_parquet("nhanes_merged.parquet", index=False)
merged.to_csv("nhanes_merged.csv", index=False)

print("\nGuardado como:")
print("- nhanes_merged.parquet")
print("- nhanes_merged.csv")